In [34]:
import os, glob, base64, getpass
import pandas as pd
import plotly.express as px
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
from jupyter_dash import JupyterDash
from CRUD_Python_Module import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()

username = os.environ.get('AAC_USERNAME', 'aacuser')
password = os.environ.get('AAC_PASSWORD') or getpass.getpass('Enter MongoDB aacuser password: ')
db = AnimalShelter(username, password)

# Verify the database is reachable rather than interpreting a failed query as zero animals.
db.client.admin.command('ping')
all_animals = db.read({})
if not all_animals:
    raise RuntimeError('No animal records returned. Check database credentials and the aac.animals collection.')

df = pd.DataFrame.from_records(all_animals).drop(columns=['_id'], errors='ignore')
print(f'Loaded {len(df):,} animals. Available fields: {list(df.columns)}')

Enter MongoDB aacuser password:  ········


Loaded 10,000 animals. Available fields: ['rec_num', 'age_upon_outcome', 'animal_id', 'animal_type', 'breed', 'color', 'date_of_birth', 'datetime', 'monthyear', 'name', 'outcome_subtype', 'outcome_type', 'sex_upon_outcome', 'location_lat', 'location_long', 'age_upon_outcome_in_weeks']


127.0.0.1 - - [22/Oct/2026 21:18:22] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [22/Oct/2026 21:18:23] "GET /_dash-component-suites/dash/deps/react@16.v2_8_1m1752168217.14.0.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Oct/2026 21:18:23] "GET /_dash-component-suites/dash/deps/prop-types@15.v2_8_1m1752168217.8.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Oct/2026 21:18:23] "GET /_dash-component-suites/dash/deps/react-dom@16.v2_8_1m1752168217.14.0.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Oct/2026 21:18:23] "GET /_dash-component-suites/dash/dash-renderer/build/dash_renderer.v2_8_1m1752168217.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Oct/2026 21:18:23] "GET /_dash-component-suites/dash_leaflet/dash_leaflet.v0_1_23m1752168237.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Oct/2026 21:18:23] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [22/Oct/2026 21:18:23] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [22/Oct/2026 21:18:23] "GET /_favicon.ico?v=2.8.1 HTTP/1.1" 200 -
127.0.0.1 - - [22/Oct/2026 21

In [35]:
WATER_BREEDS = ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']
MOUNTAIN_BREEDS = ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 'Siberian Husky', 'Rottweiler']
TRACKING_BREEDS = ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever', 'Bloodhound', 'Rottweiler']

def rescue_query(category):
    if category == 'Water Rescue':
        return {'animal_type': 'Dog', 'breed': {'$in': WATER_BREEDS},
                'sex_upon_outcome': 'Intact Female', 'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}}
    if category == 'Mountain/Wilderness Rescue':
        return {'animal_type': 'Dog', 'breed': {'$in': MOUNTAIN_BREEDS},
                'sex_upon_outcome': 'Intact Male', 'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}}
    if category == 'Disaster/Individual Tracking':
        return {'animal_type': 'Dog', 'breed': {'$in': TRACKING_BREEDS},
                'sex_upon_outcome': 'Intact Male', 'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}}
    return {}

def get_records(category):
    records = db.read(rescue_query(category))
    return pd.DataFrame.from_records(records).drop(columns=['_id'], errors='ignore').to_dict('records') if records else []

# Find a logo supplied in Codio's code_files folder.
logo_candidates = [p for pattern in ('*Grazioso*logo*.png', '*grazioso*.png', '*Logo*.png', '*logo*.png', 'code_files/*logo*.png') for p in glob.glob(pattern)]
logo = None
if logo_candidates:
    with open(logo_candidates[0], 'rb') as f:
        logo = 'data:image/png;base64,' + base64.b64encode(f.read()).decode('ascii')
    print('Logo found:', logo_candidates[0])
else:
    print('Grazioso Salvare logo not found: copy the supplied PNG next to this notebook before taking screenshots.')

Logo found: Grazioso Salvare Logo.png


In [36]:
app = JupyterDash(__name__)
header = [html.Img(src=logo, style={'height': '85px', 'objectFit': 'contain'})] if logo else [html.Div('Grazioso Salvare — ADD OFFICIAL LOGO', style={'color': '#a33'})]
app.layout = html.Div([
    html.Div(header + [html.H1('Grazioso Salvare Rescue Animal Dashboard'), html.H3('Jordan Valladares | CS 340 Project Two')], style={'textAlign': 'center'}),
    html.Hr(),
    html.Label('Filter rescue candidates:'),
    dcc.RadioItems(id='filter-type', options=[{'label': x, 'value': x} for x in ['Reset', 'Water Rescue', 'Mountain/Wilderness Rescue', 'Disaster/Individual Tracking']], value='Reset', inline=True, style={'marginBottom': '18px'}),
    html.Div(id='results-count'),
    dash_table.DataTable(id='datatable-id',
        columns=[{'name': c, 'id': c, 'selectable': True} for c in df.columns],
        data=df.to_dict('records'), page_size=12, page_action='native',
        sort_action='native', filter_action='native',
        row_selectable='single', selected_rows=[],
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'minWidth': '100px', 'maxWidth': '250px', 'overflow': 'hidden', 'textOverflow': 'ellipsis'},
        style_header={'fontWeight': 'bold', 'backgroundColor': '#e7f2f5'}),
    html.Div([html.Div(id='graph-id', style={'width': '50%', 'minWidth': '350px'}),
              html.Div(id='map-id', style={'width': '50%', 'minWidth': '350px'})],
             style={'display': 'flex', 'flexWrap': 'wrap'}),
], style={'padding': '18px', 'fontFamily': 'Arial, sans-serif'})

@app.callback([Output('datatable-id', 'data'), Output('datatable-id', 'selected_rows'), Output('results-count', 'children')], Input('filter-type', 'value'))
def update_dashboard(category):
    records = get_records(category)
    return records, [], f'{len(records):,} matching animal records — {category}'

@app.callback(Output('graph-id', 'children'), Input('datatable-id', 'derived_virtual_data'))
def update_graphs(rows):
    if rows is None: rows = df.to_dict('records')
    if not rows: return html.P('No animals match this selection.')
    current = pd.DataFrame(rows)
    if 'breed' not in current.columns: return html.P('Breed field is unavailable.')
    counts = current['breed'].fillna('Unknown').value_counts().head(10).reset_index()
    counts.columns = ['Breed', 'Count']
    fig = px.bar(counts, x='Count', y='Breed', orientation='h', title='Top 10 breeds in current results')
    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, margin={'l': 12, 'r': 12, 't': 50, 'b': 12})
    return dcc.Graph(figure=fig, config={'displayModeBar': False})

@app.callback(Output('map-id', 'children'), [Input('datatable-id', 'derived_virtual_data'), Input('datatable-id', 'derived_virtual_selected_rows')])
def update_map(rows, selected_rows):
    if rows is None: rows = df.to_dict('records')
    if not rows: return html.P('No animals available to map.')
    current = pd.DataFrame(rows)
    if not {'location_lat', 'location_long'}.issubset(current.columns):
        return html.P('Map requires location_lat and location_long fields.')
    idx = selected_rows[0] if selected_rows and selected_rows[0] < len(current) else 0
    row = current.iloc[idx]
    lat = pd.to_numeric(row.get('location_lat'), errors='coerce')
    lon = pd.to_numeric(row.get('location_long'), errors='coerce')
    if pd.isna(lat) or pd.isna(lon): return html.P('Selected animal has no valid coordinates. Select another row.')
    name = str(row.get('name') or 'Unnamed animal')
    breed = str(row.get('breed') or 'Unknown breed')
    return dl.Map(center=[float(lat), float(lon)], zoom=11, style={'width': '100%', 'height': '460px'}, children=[
        dl.TileLayer(), dl.Marker(position=[float(lat), float(lon)], children=[dl.Tooltip(breed), dl.Popup([html.H4(name), html.P(breed)])])])

@app.callback(Output('datatable-id', 'style_data_conditional'), Input('datatable-id', 'selected_columns'))
def highlight_columns(columns):
    return [{'if': {'column_id': c}, 'backgroundColor': '#d2f3ff'} for c in (columns or [])]

In [37]:
# Run in Codio. If port 8050 is occupied, change it to 8051.
app.run_server(mode='external', port=8050, debug=False)

 * Running on http://127.0.0.1:8050/ (Press CTRL+C to quit)
127.0.0.1 - - [22/Oct/2026 21:17:26] "GET /_alive_40464e06-13e1-4196-8fe4-cbc76958c75a HTTP/1.1" 200 -


Dash app running on https://edwardarnold-sonarelectra-3000.codio.io/proxy/8050/
